## Ontology Builder

In [1]:
# pip install pandas
# pip install rdflib

In [ ]:
import pandas as pd
from rdflib import Graph, Literal, Namespace, RDF, RDFS, URIRef
from rdflib.namespace import OWL, XSD
import re

# --- 1. SETUP (Same as before) ---
BMO = Namespace("http://www.bridge-management-ontology.org/bmo#")
g = Graph()
g.bind("bmo", BMO)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)


# --- 2. DEFINE LEVEL 1 CLASSES & OBJECT PROPERTIES (Same as before) ---
level1_classes = ["Structure", "Element", "Maintenance", "Location", "Traffic", "Weather"]
bridge_class = BMO.Bridge
g.add((bridge_class, RDF.type, OWL.Class))
g.add((bridge_class, RDFS.label, Literal("Bridge")))
g.add((bridge_class, RDFS.comment, Literal("Represents a single bridge structure, the central point of the ontology.")))

for class_name in level1_classes:
    class_uri = BMO[class_name]
    g.add((class_uri, RDF.type, OWL.Class))
    g.add((class_uri, RDFS.label, Literal(class_name)))
    prop_name_str = f"has{class_name}"
    prop_uri = BMO[prop_name_str]
    g.add((prop_uri, RDF.type, OWL.ObjectProperty))
    g.add((prop_uri, RDFS.label, Literal(prop_name_str)))
    g.add((prop_uri, RDFS.domain, bridge_class))
    g.add((prop_uri, RDFS.range, class_uri))


# --- 3. DEFINE LEVEL 2 DATA PROPERTIES (Same as before) ---
# This helper function is now more robust for creating property URIs
def create_data_property(name, domain_class, data_type=XSD.string, comment=""):
    """Creates an OWL.DatatypeProperty and adds it to the graph."""
    prop_name_camel = ''.join(x.title() for x in re.split(r'[\s\(\)./]', name) if x)
    prop_name_camel = prop_name_camel[0].lower() + prop_name_camel[1:]
    
    prop_uri = BMO[prop_name_camel]
    g.add((prop_uri, RDF.type, OWL.DatatypeProperty))
    g.add((prop_uri, RDFS.label, Literal(name)))
    g.add((prop_uri, RDFS.domain, BMO[domain_class]))
    g.add((prop_uri, RDFS.range, data_type))
    if comment:
        g.add((prop_uri, RDFS.comment, Literal(comment)))

# Defining all properties again for completeness
structure_props = {"Structure_Number": XSD.string, "Main_Span_Material": XSD.string, "Deck_Structure_Type": XSD.string, "Year_Built": XSD.gYear, "Structure_Length_ft": XSD.decimal, "Deck_Area_sqft": XSD.decimal}
for name, dtype in structure_props.items(): create_data_property(name, "Structure", dtype)

element_props = {"Element_Category": XSD.string, "Element_Name": XSD.string, "Total_Quantity": XSD.decimal, "Condition_State_1": XSD.integer, "Condition_State_2": XSD.integer, "Condition_State_3": XSD.integer, "Condition_State_4": XSD.integer}
for name, dtype in element_props.items(): create_data_property(name, "Element", dtype)

maintenance_props = {"Inspection_Date": XSD.date, "Inspection_Frequency_months": XSD.integer, "Bridge_Condition": XSD.string, "Bridge_Improvement_Cost": XSD.decimal}
for name, dtype in maintenance_props.items(): create_data_property(name, "Maintenance", dtype)

location_props = {"State": XSD.string, "County": XSD.string, "City": XSD.string, "Latitude": XSD.decimal, "Longitude": XSD.decimal}
for name, dtype in location_props.items(): create_data_property(name, "Location", dtype)

traffic_props = {"ADT": XSD.integer, "ADTT": XSD.integer, "Year_of_ADT": XSD.gYear}
for name, dtype in traffic_props.items(): create_data_property(name, "Traffic", dtype)

weather_props = {"Avg_Daily_Temp_C": XSD.decimal, "Total_Precipitation_mm": XSD.decimal, "Number_of_Snowfall_Days": XSD.integer}
for name, dtype in weather_props.items(): create_data_property(name, "Weather", dtype)


# --- 4. DATA MAPPING AND HELPER FUNCTION ---
# This dictionary maps the clear property names to the column names in your CSV file.
COLUMN_MAP = {
    # Structure
    "Structure_Number": "STRUCTURE_NUMBER", "Main_Span_Material": "NBI_43A - Main Span Material", "Deck_Structure_Type": "NBI_107 - Deck Structure Type Code", "Year_Built": "NBI_27 - Year Built", "Structure_Length_ft": "NBI_49 - Structure Length (ft.)", "Deck_Area_sqft": "NBI_CAT29 - Deck Area (sq. ft.)",
    # Element
    "Element_Category": "ELEMENT_CATEGORY", "Element_Name": "ELEMENT_NAME", "Total_Quantity": "TOTAL_QUANTITY", "Condition_State_1": "CONDITION_STATE_1", "Condition_State_2": "CONDITION_STATE_2", "Condition_State_3": "CONDITION_STATE_3", "Condition_State_4": "CONDITION_STATE_4",
    # Maintenance
    "Inspection_Date": "NBI_90 - Inspection Date", "Inspection_Frequency_months": "NBI_91 - Designated Inspection Frequency", "Bridge_Condition": "NBI_CAT10 - Bridge Condition", "Bridge_Improvement_Cost": "NBI_94 - Bridge Improvement Cost",
    # Location
    "State": "State", "County": "County", "City": "City", "Latitude": "LATITUDE", "Longitude": "LONGITUDE",
    # Traffic
    "ADT": "NBI_29 - Average Daily Traffic", "ADTT": "NBI_109 - Average Daily Truck Traffic (%)", "Year_of_ADT": "NBI_30 - Year of Average Daily Traffic",
    # Weather (Note: These may not be in your file, the script will handle it if they are missing)
    "Avg_Daily_Temp_C": "NBI_Avg Daily Temperature", "Total_Precipitation_mm": "NBI_Total Precipitation", "Number_of_Snowfall_Days": "NBI_Number of Snowfall Days",
}

def add_literal_if_exists(graph, subject, prop_label, data_row, data_type):
    """Checks if a column exists and its value is not null, then adds the triple."""
    prop_name_camel = ''.join(x.title() for x in re.split(r'[\s\(\)./]', prop_label) if x)
    prop_name_camel = prop_name_camel[0].lower() + prop_name_camel[1:]
    prop_uri = BMO[prop_name_camel]
    
    col_name = COLUMN_MAP.get(prop_label)
    if col_name and col_name in data_row and pd.notna(data_row[col_name]):
        value = data_row[col_name]
        graph.add((subject, prop_uri, Literal(value, datatype=data_type)))

# --- 5. POPULATE ONTOLOGY FROM CSV ---
try:
    df = pd.read_csv("NBE_and_NBI_OK.csv", low_memory=False)

    # First, create individuals for each unique bridge and its 1-to-1 properties
    unique_bridges = df.drop_duplicates(subset=['STRUCTURE_NUMBER'])
    for index, row in unique_bridges.iterrows():
        struct_num = row['STRUCTURE_NUMBER']
        bridge_uri = BMO[f"Bridge_{struct_num}"]
        g.add((bridge_uri, RDF.type, bridge_class))

        # Create and link individuals for each category
        for class_name, props in [("Structure", structure_props), ("Maintenance", maintenance_props), ("Location", location_props), ("Traffic", traffic_props), ("Weather", weather_props)]:
            individual_uri = BMO[f"{class_name}_{struct_num}"]
            g.add((individual_uri, RDF.type, BMO[class_name]))
            g.add((bridge_uri, BMO[f"has{class_name}"], individual_uri))
            
            # Populate data properties for this individual
            for prop_label, prop_type in props.items():
                add_literal_if_exists(g, individual_uri, prop_label, row, prop_type)

    # Second, iterate through all rows to create Element individuals (1-to-many)
    for index, row in df.iterrows():
        struct_num = row['STRUCTURE_NUMBER']
        if pd.isna(struct_num): continue
        
        bridge_uri = BMO[f"Bridge_{struct_num}"]
        
        # Create a unique URI for each element instance, using the dataframe index
        element_uri = BMO[f"Element_{struct_num}_{index}"]
        g.add((element_uri, RDF.type, BMO.Element))
        g.add((bridge_uri, BMO.hasElement, element_uri))
        
        # Populate data properties for the Element
        for prop_label, prop_type in element_props.items():
            add_literal_if_exists(g, element_uri, prop_label, row, prop_type)
    
    print("Successfully processed the dataset.")

except FileNotFoundError:
    print("ERROR: Could not find 'NBE_and_NBI_OK.csv'. Make sure it's in the same directory.")
except Exception as e:
    print(f"An error occurred: {e}")


# --- 6. SERIALIZE AND SAVE THE POPULATED ONTOLOGY ---
try:
    g.serialize(destination="bridge_ontology_populated.owl", format="xml")
    print("\nSuccess! Ontology file 'bridge_ontology_populated.owl' has been created.")
    print("You can now open this file in Protégé.")
except Exception as e:
    print(f"An error occurred while saving the file: {e}")
